# Notebook 03: Meta-Learners (S, T, X)

This notebook implements and compares the classical meta-learner family for CATE estimation.

Notation refresher:
- `X`: pre-treatment features
- `T`: binary treatment (any email vs no email)
- `Y`: conversion outcome
- `mu1(x)=E[Y|T=1,X=x]`, `mu0(x)=E[Y|T=0,X=x]`
- `tau(x)=mu1(x)-mu0(x)`

In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

if Path.cwd().name == 'notebooks':
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import set_seed, ensure_output_dirs, save_plot
from src.preprocessing import load_hillstrom_data, basic_cleaning, split_features_outcomes_treatment, create_train_valid_test_split
from src.meta_learners import SLearner, TLearner, XLearner
from src.evaluation import qini_curve, qini_coefficient, auuc_score, uplift_curve, cumulative_gain_curve, uplift_by_decile

SEED = 42
set_seed(SEED)
OUT_DIRS = ensure_output_dirs(PROJECT_ROOT)
DATA_PATH = PROJECT_ROOT / 'data' / 'hillstrom.csv'

In [ ]:
df = basic_cleaning(load_hillstrom_data(str(DATA_PATH)))
X, y, t = split_features_outcomes_treatment(df, outcome_col='conversion', treatment_col='treatment')
splits = create_train_valid_test_split(X, y, t, test_size=0.2, valid_size=0.2, random_state=SEED, stratify=True)
feature_names = X.columns.tolist()
print('Split sizes:', {k: len(v) for k, v in splits.items() if k.startswith('X_')})

## 1) Fit S-Learner, T-Learner, X-Learner

In [ ]:
s_learner = SLearner(random_state=SEED).fit(splits['X_train'], splits['t_train'], splits['y_train'])
t_learner = TLearner(random_state=SEED).fit(splits['X_train'], splits['t_train'], splits['y_train'])
x_learner = XLearner(random_state=SEED).fit(splits['X_train'], splits['t_train'], splits['y_train'])

scores = {
    's_learner': s_learner.predict_uplift(splits['X_test']),
    't_learner': t_learner.predict_uplift(splits['X_test']),
    'x_learner': x_learner.predict_uplift(splits['X_test']),
}
for k, v in scores.items():
    print(k, pd.Series(v).describe().to_dict())

## 2) Evaluate ranking quality

In [ ]:
rows = []
for model_name, score in scores.items():
    rows.append({
        'model': model_name,
        'qini': qini_coefficient(splits['y_test'], splits['t_test'], score),
        'auuc': auuc_score(splits['y_test'], splits['t_test'], score),
    })
metrics = pd.DataFrame(rows).sort_values('qini', ascending=False).reset_index(drop=True)
display(metrics)
metrics.to_csv(OUT_DIRS['tables'] / 'nb03_meta_learner_metrics.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for model_name, score in scores.items():
    c = qini_curve(splits['y_test'], splits['t_test'], score)
    ax.plot(c['fraction_targeted'], c['incremental_outcomes'], label=model_name)
ax.set_title('Qini Curve Comparison: S/T/X Learners')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative incremental conversions')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb03_qini_meta_learner_comparison.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for model_name, score in scores.items():
    c = uplift_curve(splits['y_test'], splits['t_test'], score)
    ax.plot(c['fraction_targeted'], c['uplift'], label=model_name)
ax.set_title('Uplift Curve Comparison: S/T/X Learners')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Estimated uplift')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb03_uplift_meta_learner_comparison.png')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for model_name, score in scores.items():
    c = cumulative_gain_curve(splits['y_test'], splits['t_test'], score)
    ax.plot(c['fraction_targeted'], c['cumulative_gain'], label=model_name)
ax.set_title('Cumulative Gain Comparison: S/T/X Learners')
ax.set_xlabel('Fraction targeted')
ax.set_ylabel('Cumulative gain')
ax.legend()
plt.tight_layout()
save_plot(fig, OUT_DIRS['figures'] / 'nb03_cumulative_gain_meta_learner_comparison.png')
plt.show()

In [ ]:
decile_rows = []
for model_name, score in scores.items():
    d = uplift_by_decile(splits['y_test'], splits['t_test'], score)
    d['model'] = model_name
    decile_rows.append(d)
deciles = pd.concat(decile_rows, ignore_index=True)
display(deciles.head(20))
deciles.to_csv(OUT_DIRS['tables'] / 'nb03_uplift_by_decile.csv', index=False)

## 3) Model interpretation note

In [ ]:
best_model = metrics.iloc[0]['model']
print('Best by Qini:', best_model)

if best_model == 't_learner' and hasattr(t_learner.model_t_, 'feature_importances_') and hasattr(t_learner.model_c_, 'feature_importances_'):
    importances = (np.asarray(t_learner.model_t_.feature_importances_) + np.asarray(t_learner.model_c_.feature_importances_)) / 2
    feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False).head(15)
    display(feat_imp)
    feat_imp.to_csv(OUT_DIRS['tables'] / 'nb03_feature_importance_top15.csv', index=False)
else:
    print('Feature importance extraction skipped for this best model family.')

Feature importance can indicate predictive drivers, but it is not a full causal explanation of treatment effect mechanisms.

## End-of-notebook summary
- S/T/X provide stronger uplift-specific ranking than naive baselines.
- Best model by ranking metrics on this split is reported above.
- Notebook 04 introduces DR-style modern causal estimation for improved robustness.